In [1]:
import numpy as np
import pandas as pd
import os 
from bs4 import BeautifulSoup

In [2]:
os.chdir("/ix/djishnu/Aaron_F/Cleaned_Proteomics_Aaron/20240906/chrombpnet/Results")

In [3]:
# Function to extract TFs with pattern and seqlet count
def extract_tf_data(html_file):
    # Read the HTML file
    with open(html_file, "r", encoding="utf-8") as f:
        html_content = f.read()

    # Parse the HTML
    soup = BeautifulSoup(html_content, "html.parser")

    # Extract table rows
    rows = soup.find_all("tr")

    # List to store extracted data
    extracted_data = []

    # Process rows to extract pattern name, seqlet count, TF names, and p-values
    for row in rows[1:]:  # Skip header row
        cols = row.find_all("td")
        
        # Extract pattern name and seqlet count
        pattern_name = cols[0].text.strip()
        seqlet_cnt = int(cols[1].text.strip())

        # Iterate through TF matches (every third column starting at index 4)
        for i in range(4, len(cols) - 1, 3):  # Ensures i+1 is valid
            tf_entry = cols[i].text.strip()
            pval = float(cols[i + 1].text.strip())

            # Extract TF names (handling "::" cases correctly)
            tf_names = tf_entry.split(".")[-1]  # Extract after last "."
            tf_list = tf_names.split("::")  # Split TF names

            # Append each TF separately while keeping the same p-value
            for tf in tf_list:
                extracted_data.append([pattern_name, seqlet_cnt, tf, pval])

    # Convert to DataFrame
    df = pd.DataFrame(extracted_data, columns=["Pattern", "Seqlet_Count", "TF_Name", "P_Value"])
    
    return df

In [4]:
# usage:
html_file = "/ix/djishnu/Aaron_F/Cleaned_Proteomics_Aaron/20240906/chrombpnet/Results/profile_motifs.html"  # Replace with actual file path
df_tf_data = extract_tf_data(html_file)

In [5]:
df_tf_data

,Pattern,Seqlet_Count,TF_Name,P_Value
0,pos_patterns.pattern_0,63657,ELK1,0.009106
1,pos_patterns.pattern_0,63657,ETV6,0.009106
2,pos_patterns.pattern_0,63657,ETV5,0.009106
3,pos_patterns.pattern_1,32845,FOSL2,0.000553
4,pos_patterns.pattern_1,32845,JUN,0.000553
...,...,...,...,...
271,neg_patterns.pattern_18,49,FOSL2,0.135617
272,neg_patterns.pattern_18,49,JUNB,0.135617
273,neg_patterns.pattern_19,26,THRA,0.775231
274,neg_patterns.pattern_19,26,Rxra,0.775231


In [6]:
# save
df_tf_data.to_csv("profile_motifs.csv", index=False)